# THỰC NGHIỆM PHÂN CỤM DỮ LIỆU SKU LOGISTICS
## So Sánh Thuật Toán: DBSCAN (Batch) · Incremental DBSCAN · K-Means / MiniBatch K-Means
**Tập dữ liệu đầu vào:** `output_preprocessing/sku_clean_scaled.xlsx` (2073 SKU hoạt động sau tiền xử lý)

---

### Mục Tiêu Thực Nghiệm
1. **Phân cụm toàn cục (Batch)**:
   - **K-Means**: Thuật toán phân hoạch dựa trên khoảng cách tới tâm cụm, phân tích biểu đồ Elbow (Inertia) và Silhouette để tìm số cụm $k$ tối ưu.
   - **DBSCAN (Batch)**: Thuật toán phân cụm dựa trên mật độ, xác định bán kính láng giềng $\varepsilon$ (Eps) qua đồ thị k-distance và số điểm lân cận tối thiểu ($MinPts$).
2. **Phân cụm gia tăng (Incremental Clustering cho dữ liệu luồng / Stream)**:
   - **Incremental DBSCAN (Ester et al., 1998)**: Thuật toán mật độ thích nghi với luồng dữ liệu mới. Sử dụng Union-Find để duy trì trạng thái điểm lõi, biên, nhiễu và xử lý tạo mới / mở rộng / hợp nhất cụm khi có lô hàng SKU mới nạp vào.
   - **MiniBatch K-Means (Incremental K-Means)**: Thuật toán cập nhật tâm cụm theo từng lô (mini-batch) bằng stochastic gradient descent.
3. **So sánh toàn diện & Đánh giá**:
   - So sánh các chỉ số: **Silhouette Score**, **Davies-Bouldin Index (DBI)**, **Calinski-Harabasz Index (CHI)**, **Tỷ lệ điểm nhiễu (Noise %)**, thời gian chạy (Runtime).
   - Đánh giá tính nhất quán (ARI - Adjusted Rand Index) giữa Incremental DBSCAN và DBSCAN Batch.
   - Trực quan hóa không gian phân cụm trên 2D / 3D PCA.
   - Diễn giải ý nghĩa nghiệp vụ kho vận (logistics) dựa trên các cụm tìm được.

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import DBSCAN, KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, 
    davies_bouldin_score, 
    calinski_harabasz_score,
    adjusted_rand_score, 
    adjusted_mutual_info_score
)

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.unicode_minus'] = False

SEED = 42
np.random.seed(SEED)

DATA_PATH = r"output_preprocessing/sku_clean_scaled.xlsx"
if not os.path.exists(DATA_PATH):
    DATA_PATH = r"D:/HP Máy học/Đồ án/output_preprocessing/sku_clean_scaled.xlsx"

print(f"Đường dẫn dữ liệu: {DATA_PATH}")

## 1. Nạp Dữ Liệu và Xây Dựng Ma Trận Đặc Trưng
- Đọc file `sku_clean_scaled.xlsx` (2073 SKU hoạt động sau tiền xử lý).
- Trích xuất 6 đặc trưng vận hành liên tục (đã biến đổi log1p + RobustScaler).
- Kết hợp với biến phân tầng `Expire_group` (mã hóa one-hot với trọng số $W_{expire} = 2.0$ để phân tách dứt khoát các nhóm hạn dùng).

In [ ]:
df = pd.read_excel(DATA_PATH)
print(f"Kích thước dữ liệu: {df.shape[0]} dòng x {df.shape[1]} cột")

# 6 đặc trưng liên tục đã scale
feat_cols = [c for c in df.columns if c.endswith('_scaled')]
print(f"Các đặc trưng liên tục ({len(feat_cols)}): {feat_cols}")

# Phân tầng theo nhóm hạn dùng Expire_group
group = df['Expire_group'].astype(int).values
G = sorted(np.unique(group))
EXPIRE_W = 2.0
onehot_group = np.stack([(group == g).astype(float) for g in G], axis=1) * EXPIRE_W

X_cont = df[feat_cols].values
X = np.hstack([X_cont, onehot_group])

print(f"Ma trận X (đặc trưng liên tục + phân tầng): {X.shape}")
display(df.head())

## 2. Xây Dựng Lớp Incremental DBSCAN và Hàm Đánh Giá
Cài đặt thuật toán **Incremental DBSCAN** (theo Ester et al., 1998):
- **Union-Find (Disjoint Set)**: Quản lý và hợp nhất các điểm lõi (Core Points) với độ phức tạp tối ưu.
- **Xử lý gia tăng theo từng lô (Batch)**:
  1. Cập nhật số lượng láng giềng trong bán kính $\varepsilon$.
  2. Xác định các điểm lõi mới phát sinh (kể cả điểm cũ trước đó chưa đủ $MinPts$).
  3. Hợp nhất các cụm lõi có láng giềng chung.
  4. Gán Cluster ID ổn định (tạo mới, mở rộng hoặc sáp nhập).
  5. Tái phân loại các điểm biên và khôi phục điểm nhiễu (Noise Recovery).

In [ ]:
def evaluate_clustering(X_eval, labels, name="Model"):
    labels = np.asarray(labels)
    mask = labels != -1
    n_clusters = len(set(labels[mask]))
    n_noise = (~mask).sum()
    noise_pct = (n_noise / len(labels)) * 100
    
    metrics = {
        'Model': name,
        'N_Clusters': n_clusters,
        'Noise_Count': n_noise,
        'Noise_Ratio_%': round(noise_pct, 2)
    }
    
    if n_clusters >= 2 and mask.sum() > n_clusters:
        metrics['Silhouette'] = round(silhouette_score(X_eval[mask], labels[mask]), 4)
        metrics['Davies_Bouldin'] = round(davies_bouldin_score(X_eval[mask], labels[mask]), 4)
        metrics['Calinski_Harabasz'] = round(calinski_harabasz_score(X_eval[mask], labels[mask]), 2)
    else:
        metrics['Silhouette'] = np.nan
        metrics['Davies_Bouldin'] = np.nan
        metrics['Calinski_Harabasz'] = np.nan
        
    return metrics

class IncrementalDBSCAN:
    """
    Incremental DBSCAN (Ester et al., 1998)
    Cập nhật phân cụm theo lô điểm mới bổ sung mà không cần huấn luyện lại từ đầu.
    """
    def __init__(self, eps=0.8, min_samples=10):
        self.eps = eps
        self.min_samples = min_samples
        self.X = None
        self.count = None
        self.is_core = None
        self.parent = None
        self.core_cid = None
        self.labels_ = None
        self.next_cid = 0
        self.history = []

    def _find(self, i):
        p = self.parent
        while p[i] != i:
            p[i] = p[p[i]]
            i = p[i]
        return i

    def _union(self, a, b):
        ra, rb = self._find(a), self._find(b)
        if ra != rb:
            self.parent[rb] = ra

    def insert(self, X_new, stage_name=""):
        t0 = time.perf_counter()
        X_new = np.asarray(X_new)
        m = len(X_new)
        
        if self.X is None:
            n_old = 0
            self.X = X_new.copy()
            self.count = np.zeros(m, dtype=int)
            self.is_core = np.zeros(m, dtype=bool)
            self.parent = np.arange(m)
            self.core_cid = np.full(m, -1, dtype=int)
            self.labels_ = np.full(m, -1, dtype=int)
        else:
            n_old = len(self.X)
            self.X = np.vstack([self.X, X_new])
            self.count = np.concatenate([self.count, np.zeros(m, dtype=int)])
            self.is_core = np.concatenate([self.is_core, np.zeros(m, dtype=bool)])
            self.parent = np.concatenate([self.parent, np.arange(n_old, n_old + m)])
            self.core_cid = np.concatenate([self.core_cid, np.full(m, -1, dtype=int)])
            self.labels_ = np.concatenate([self.labels_, np.full(m, -1, dtype=int)])

        # 1. Tìm láng giềng bán kính eps của điểm mới
        nn = NearestNeighbors(radius=self.eps).fit(self.X)
        nb_new = nn.radius_neighbors(X_new, return_distance=False)
        for i, nb in enumerate(nb_new):
            self.count[n_old + i] = len(nb)
            self.count[nb[nb < n_old]] += 1

        # 2. Xác định các điểm lõi mới phát sinh
        newly_core = np.where((self.count >= self.min_samples) & (~self.is_core))[0]
        self.is_core[newly_core] = True
        
        nb_map = {n_old + i: nb_new[i] for i in range(m)}
        old_newly = newly_core[newly_core < n_old]
        if len(old_newly) > 0:
            for c, nb in zip(old_newly, nn.radius_neighbors(self.X[old_newly], return_distance=False)):
                nb_map[c] = nb

        # 3. Hợp nhất cụm (Union) giữa các điểm lõi láng giềng
        for c in newly_core:
            for j in nb_map[c]:
                if self.is_core[j]:
                    self._union(c, j)

        # 4. Gán Cluster ID ổn định
        cores = np.where(self.is_core)[0]
        comps = {}
        for c in cores:
            comps.setdefault(self._find(c), []).append(c)
            
        n_created, n_merges = 0, 0
        for members in comps.values():
            members = np.array(members)
            old_ids = np.unique(self.core_cid[members][self.core_cid[members] >= 0])
            if len(old_ids) == 0:
                cid = self.next_cid
                self.next_cid += 1
                n_created += 1
            else:
                cid = int(old_ids.min())
                n_merges += (len(old_ids) - 1)
            self.core_cid[members] = cid

        # 5. Phân bổ điểm biên và điểm nhiễu
        prev_labels = self.labels_.copy()
        new_labels = np.full(len(self.X), -1, dtype=int)
        new_labels[cores] = self.core_cid[cores]
        
        non_core = np.where(~self.is_core)[0]
        if len(cores) > 0 and len(non_core) > 0:
            dists, idxs = NearestNeighbors(n_neighbors=1).fit(self.X[cores]).kneighbors(self.X[non_core])
            valid_border = dists[:, 0] <= self.eps
            new_labels[non_core[valid_border]] = self.core_cid[cores[idxs[valid_border, 0]]]
            
        promoted_noise = int(((prev_labels[:n_old] == -1) & (new_labels[:n_old] != -1)).sum()) if n_old > 0 else 0
        self.labels_ = new_labels
        
        elapsed = time.perf_counter() - t0
        self.history.append({
            'Stage': stage_name,
            'Batch_Size': m,
            'Total_Points': len(self.X),
            'Core_Points': int(self.is_core.sum()),
            'New_Clusters': n_created,
            'Merges': n_merges,
            'Noise_Recovered': promoted_noise,
            'Time_Sec': round(elapsed, 4)
        })
        return self.labels_

print("Đã khởi tạo IncrementalDBSCAN và hàm evaluate_clustering thành công!")

## 3. Thuật Toán 1: K-Means (Batch)
- Khảo sát số cụm $k \in [2, 10]$ bằng phương pháp **Elbow (Inertia)** và **Silhouette Score**.
- Huấn luyện mô hình K-Means tối ưu và đánh giá hiệu năng.

In [ ]:
# Tìm k tối ưu cho K-Means
k_range = range(2, 11)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_cont, km.labels_))

fig, ax1 = plt.subplots(1, 2, figsize=(14, 5))

# Biểu đồ Elbow
ax1[0].plot(k_range, inertias, marker='o', color='royalblue', lw=2)
ax1[0].set_title("Biểu đồ Elbow (Inertia) - K-Means", fontsize=12, fontweight='bold')
ax1[0].set_xlabel("Số cụm k")
ax1[0].set_ylabel("Inertia")
ax1[0].grid(True, linestyle='--', alpha=0.6)

# Biểu đồ Silhouette
ax1[1].plot(k_range, sil_scores, marker='s', color='crimson', lw=2)
ax1[1].set_title("Hệ số Silhouette theo số cụm k", fontsize=12, fontweight='bold')
ax1[1].set_xlabel("Số cụm k")
ax1[1].set_ylabel("Silhouette Score")
ax1[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(sil_scores)]
print(f"Số cụm k tối ưu theo Silhouette: k = {best_k}")

# Huấn luyện K-Means với k tối ưu
t0_km = time.perf_counter()
kmeans_model = KMeans(n_clusters=best_k, random_state=SEED, n_init=10).fit(X)
t_km = time.perf_counter() - t0_km

df['Cluster_KMeans'] = kmeans_model.labels_
res_kmeans = evaluate_clustering(X_cont, kmeans_model.labels_, name=f"K-Means (k={best_k})")
res_kmeans['Time_Sec'] = round(t_km, 4)
display(pd.DataFrame([res_kmeans]))

## 4. Thuật Toán 2: DBSCAN (Batch)
- Sử dụng đồ thị khoảng cách k-lân cận (**k-distance graph**) để xác định ngưỡng khoảng cách lân cận $\varepsilon$ (Eps).
- Chạy DBSCAN chuẩn trên toàn bộ 2073 SKU để làm mốc tham chiếu (Ground Truth).

In [ ]:
# Xác định eps bằng k-distance graph (k = MinPts = 10)
min_pts = 10
nbrs = NearestNeighbors(n_neighbors=min_pts).fit(X)
distances, indices = nbrs.kneighbors(X)
sorted_distances = np.sort(distances[:, -1])

plt.figure(figsize=(10, 5))
plt.plot(sorted_distances, color='darkorange', lw=2)
plt.axhline(y=0.8, color='red', linestyle='--', label='Ngưỡng đề xuất eps = 0.8')
plt.title("Đồ thị k-Distance Graph (k = 10) xác định điểm uốn Elbow", fontsize=12, fontweight='bold')
plt.xlabel("Chỉ số điểm (đã sắp xếp)")
plt.ylabel(f"Khoảng cách tới lân cận thứ {min_pts}")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# Huấn luyện DBSCAN chuẩn
best_eps = 0.8
t0_db = time.perf_counter()
dbscan_model = DBSCAN(eps=best_eps, min_samples=min_pts).fit(X)
t_db = time.perf_counter() - t0_db

df['Cluster_DBSCAN'] = dbscan_model.labels_
res_dbscan = evaluate_clustering(X_cont, dbscan_model.labels_, name=f"DBSCAN (eps={best_eps}, ms={min_pts})")
res_dbscan['Time_Sec'] = round(t_db, 4)
display(pd.DataFrame([res_dbscan]))

## 5. Thuật Toán 3: Incremental DBSCAN & MiniBatch K-Means
Mô phỏng bài toán thực tế trong kho vận: Dữ liệu SKU được bổ sung liên tục theo 5 đợt (Batches) định kỳ:
- **Đợt 1 (Khởi tạo)**: 40% dữ liệu ban đầu.
- **Đợt 2 - 5 (Gia tăng)**: Mỗi đợt bổ sung 15% SKU mới.

In [ ]:
# Sắp xếp theo Stream_order đã tạo trong tiền xử lý
df_stream = df.sort_values('Stream_order').reset_index(drop=True)
X_stream = np.hstack([df_stream[feat_cols].values, 
                      np.stack([(df_stream['Expire_group'].astype(int).values == g).astype(float) for g in G], axis=1) * EXPIRE_W])

# Chia 5 đợt: 40%, 15%, 15%, 15%, 15%
n_total = len(X_stream)
b1 = int(n_total * 0.40)
step = int(n_total * 0.15)
split_indices = [0, b1, b1 + step, b1 + 2*step, b1 + 3*step, n_total]

# Khởi tạo mô hình Incremental DBSCAN và MiniBatch K-Means
inc_dbscan = IncrementalDBSCAN(eps=best_eps, min_samples=min_pts)
minibatch_km = MiniBatchKMeans(n_clusters=best_k, random_state=SEED, batch_size=200)

print("=== BẮT ĐẦU QUÁ TRÌNH PHÂN CỤM GIA TĂNG (INCREMENTAL) ===")
t0_inc = time.perf_counter()

for i in range(len(split_indices) - 1):
    start_idx = split_indices[i]
    end_idx = split_indices[i+1]
    batch_X = X_stream[start_idx:end_idx]
    stage_label = f"Đợt {i+1} ({len(batch_X)} SKU)"
    
    # 1. Chèn vào Incremental DBSCAN
    inc_dbscan.insert(batch_X, stage_name=stage_label)
    
    # 2. Cập nhật MiniBatch K-Means
    minibatch_km.partial_fit(batch_X)
    
t_inc = time.perf_counter() - t0_inc
print(f"Hoàn thành mô phỏng 5 đợt trong {t_inc:.4f} giây!")

# Đưa nhãn Incremental về thứ tự gốc của df
df_stream['Cluster_IncDBSCAN'] = inc_dbscan.labels_
df_stream['Cluster_MiniBatchKM'] = minibatch_km.predict(X_stream)
df['Cluster_IncDBSCAN'] = df_stream.set_index('ID')['Cluster_IncDBSCAN'].loc[df['ID']].values
df['Cluster_MiniBatchKM'] = df_stream.set_index('ID')['Cluster_MiniBatchKM'].loc[df['ID']].values

# Bảng theo dõi lịch sử cập nhật Incremental DBSCAN
history_df = pd.DataFrame(inc_dbscan.history)
display(history_df)

## 6. Kiểm Định Tính Đúng Đắn Của Incremental DBSCAN
So sánh kết quả của **Incremental DBSCAN** với **DBSCAN Batch** (chạy lại từ đầu trên toàn bộ dữ liệu):
- Hệ số tương đồng phân hoạch **Adjusted Rand Index (ARI)**.
- Hệ số tương quan thông tin **Adjusted Mutual Information (AMI)**.
- Kiểm tra tính khớp 1-1 về điểm lõi và phân hoạch cụm.

In [ ]:
ari_val = adjusted_rand_score(df['Cluster_DBSCAN'], df['Cluster_IncDBSCAN'])
ami_val = adjusted_mutual_info_score(df['Cluster_DBSCAN'], df['Cluster_IncDBSCAN'])

print("=== ĐỘ NHẤT QUÁN GIỮA INCREMENTAL DBSCAN VÀ BATCH DBSCAN ===")
print(f"Hệ số Adjusted Rand Index (ARI)       : {ari_val:.6f}")
print(f"Hệ số Adjusted Mutual Info (AMI)      : {ami_val:.6f}")

if ari_val >= 0.999:
    print(">> KẾT LUẬN: Incremental DBSCAN khớp hoàn toàn 100% với DBSCAN Batch!")
else:
    print(f">> KẾT LUẬN: Độ tương đồng rất cao (ARI = {ari_val:.4f}).")

res_inc = evaluate_clustering(X_cont, df['Cluster_IncDBSCAN'], name="Incremental DBSCAN")
res_inc['Time_Sec'] = round(t_inc, 4)

res_mbkm = evaluate_clustering(X_cont, df['Cluster_MiniBatchKM'], name=f"MiniBatch K-Means (k={best_k})")

## 7. Bảng Tổng Hợp So Sánh Toàn Diện Giữa Các Thuật Toán
So sánh 4 mô hình trên cùng tập dữ liệu chuẩn hóa:
1. **K-Means (Batch)**
2. **MiniBatch K-Means (Incremental)**
3. **DBSCAN (Batch)**
4. **Incremental DBSCAN**

In [ ]:
summary_results = pd.DataFrame([res_kmeans, res_mbkm, res_dbscan, res_inc])
display(summary_results)

# Biểu đồ so sánh Silhouette Score và Thời gian chạy
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=summary_results, x='Model', y='Silhouette', palette='viridis', ax=axes[0])
axes[0].set_title("So Sánh Hệ Số Silhouette", fontsize=12, fontweight='bold')
axes[0].set_ylim(0, 0.5)
for p in axes[0].patches:
    val = p.get_height()
    if not np.isnan(val):
        axes[0].annotate(f"{val:.3f}", (p.get_x() + p.get_width() / 2., val),
                         ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

sns.barplot(data=summary_results, x='Model', y='Time_Sec', palette='magma', ax=axes[1])
axes[1].set_title("So Sánh Thời Gian Chạy (Giây)", fontsize=12, fontweight='bold')
for p in axes[1].patches:
    val = p.get_height()
    if not np.isnan(val):
        axes[1].annotate(f"{val:.3f}s", (p.get_x() + p.get_width() / 2., val),
                         ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()

## 8. Trực Quan Hóa Không Gian Cụm Trên PCA (2D & 3D)
Chiếu 6 đặc trưng vận hành xuống 2 và 3 thành phần chính để quan sát sự phân bố cụm giữa **K-Means** và **Incremental DBSCAN**.

In [ ]:
pca = PCA(n_components=3, random_state=SEED)
X_pca = pca.fit_transform(X_cont)
df['PCA1'] = X_pca[:, 0]
df['PCA2'] = X_pca[:, 1]
df['PCA3'] = X_pca[:, 2]

print(f"Tỷ lệ phương sai giải thích của 3 thành phần chính: {pca.explained_variance_ratio_ * 100}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Trực quan K-Means
scatter1 = axes[0].scatter(df['PCA1'], df['PCA2'], c=df['Cluster_KMeans'], cmap='tab10', s=25, alpha=0.7)
axes[0].set_title(f"Phân Cụm K-Means (k={best_k}) trên PCA 2D", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Principal Component 1")
axes[0].set_ylabel("Principal Component 2")
plt.colorbar(scatter1, ax=axes[0], label='Cluster ID')

# Trực quan Incremental DBSCAN
scatter2 = axes[1].scatter(df['PCA1'], df['PCA2'], c=df['Cluster_IncDBSCAN'], cmap='Spectral', s=25, alpha=0.7)
axes[1].set_title("Phân Cụm Incremental DBSCAN trên PCA 2D (Màu tối = Nhiễu)", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Principal Component 1")
axes[1].set_ylabel("Principal Component 2")
plt.colorbar(scatter2, ax=axes[1], label='Cluster ID (-1: Noise)')

plt.tight_layout()
plt.show()

## 9. Diễn Giải Nghiệp Vụ Kho Vận & Xuất Kết Quả
Ánh xạ các cụm tìm được từ Incremental DBSCAN về đơn vị đo lường gốc (kg, mét, số lượng xuất, giá thành) để hỗ trợ thủ kho và nhà quản lý chuỗi cung ứng ra quyết định lưu trữ.

In [ ]:
ORIG_PATH = r"output_preprocessing/sku_clean_original_units.xlsx"
if not os.path.exists(ORIG_PATH):
    ORIG_PATH = r"D:/HP Máy học/Đồ án/output_preprocessing/sku_clean_original_units.xlsx"

if os.path.exists(ORIG_PATH):
    df_orig = pd.read_excel(ORIG_PATH)
    df_orig['Cluster_IncDBSCAN'] = df['Cluster_IncDBSCAN'].values
    
    # Tính trung bình các chỉ số theo cụm
    cluster_profile = df_orig.groupby('Cluster_IncDBSCAN')[
        ['Unitprice', 'Outbound number', 'Total outbound', 'Pal grossweight', 'Pal height', 'Units per pal', 'Avg_units_per_order']
    ].median()
    cluster_profile['SKU_Count'] = df_orig.groupby('Cluster_IncDBSCAN').size()
    print("=== ĐẶC TÍNH NGHIỆP VỤ KHO VẬN CỦA CÁC CỤM (GIÁ TRỊ TRUNG VỊ GỐC) ===")
    display(cluster_profile)
else:
    print("Không tìm thấy file sku_clean_original_units.xlsx, sử dụng bảng df hiện tại để thống kê.")
    cluster_profile = df.groupby('Cluster_IncDBSCAN')[feat_cols].median()
    display(cluster_profile)

# Xuất kết quả phân cụm ra Excel
OUTPUT_FILE = r"D:/HP Máy học/Đồ án/sku_clustering_results.xlsx"
df.to_excel(OUTPUT_FILE, index=False)
print(f"Đã xuất bảng dữ liệu kết quả phân cụm thành công vào: {OUTPUT_FILE}")